# BarClean r02 learned-constraint diagnostics

Diagnose why stage-2 `tool_pitch` and stage-4 `tool_roll` were classified inactive by the latest demo-balanced pooled MAP run. Stage numbering in the tables is user-facing (1–5).

In [3]:
from pathlib import Path
import json
import numpy as np

root = Path('/home/baiyu/LearnStageConstraints')
archive_path = root / 'robot/stage_cons_iiwa14/data/processed/demo_r02_auto/demo_r02_12_demos_5hz_training.npz'
learned_path = root / 'outputs/map_balanced_pooled/BarClean/method_seed_000/learned_constraints.json'
true_path = root / 'robot/stage_cons_iiwa14/ros_ws/src/stage_constraint_planner/config/bar_clean_true.json'

data = np.load(archive_path, allow_pickle=False)
features = np.asarray(data['features'], dtype=float)
feature_names = [str(x) for x in data['feature_names'].tolist()]
bounds = np.asarray(data['coarse_bounds_indices'], dtype=int)
source_ids = np.asarray(data['source_demo_ids'], dtype=int)
learned = json.loads(learned_path.read_text())
true_cfg = json.loads(true_path.read_text())
true_targets = {(int(t['stage']), str(t['feature_name'])): float(t['value']) for t in true_cfg['constraint_terms'] if t['semantics'] == 'target_value'}

pairs = [(1, 'tool_pitch'), (1, 'tool_roll'), (3, 'tool_pitch'), (3, 'tool_roll')]
rows = []
for stage, feature_name in pairs:
    j = feature_names.index(feature_name)
    target = true_targets[(stage, feature_name)]
    for demo_index, source_id in enumerate(source_ids):
        start, end = bounds[demo_index, stage:stage + 2]
        values = features[start:end, j]
        split = max(1, len(values) // 2)
        rows.append({
            'user_stage': stage + 1,
            'feature': feature_name,
            'source_demo': int(source_id),
            'n': len(values),
            'mean_deg': float(np.rad2deg(np.mean(values))),
            'std_deg': float(np.rad2deg(np.std(values))),
            'min_deg': float(np.rad2deg(np.min(values))),
            'max_deg': float(np.rad2deg(np.max(values))),
            'mean_error_deg': float(np.rad2deg(np.mean(values) - target)),
            'second_minus_first_deg': float(np.rad2deg(np.mean(values[split:]) - np.mean(values[:split]))),
        })

summary_rows = []
for user_stage, feature_name in [(2, 'tool_pitch'), (2, 'tool_roll'), (4, 'tool_pitch'), (4, 'tool_roll')]:
    group = [r for r in rows if r['user_stage'] == user_stage and r['feature'] == feature_name]
    demo_means = np.asarray([r['mean_deg'] for r in group])
    within = np.asarray([r['std_deg'] for r in group])
    drift = np.asarray([r['second_minus_first_deg'] for r in group])
    summary_rows.append({
        'user_stage': user_stage,
        'feature': feature_name,
        'demo_mean_mean_deg': float(np.mean(demo_means)),
        'between_demo_std_deg': float(np.std(demo_means)),
        'demo_mean_range_deg': float(np.ptp(demo_means)),
        'median_within_demo_std_deg': float(np.median(within)),
        'max_within_demo_std_deg': float(np.max(within)),
        'median_abs_drift_deg': float(np.median(np.abs(drift))),
    })

mode_lookup = {(int(x['stage']), x['feature_name']): x for x in learned['feature_stage_modes']}
mode_rows = []
for stage, feature_name in pairs:
    item = mode_lookup[(stage, feature_name)]
    mode_rows.append({
        'user_stage': stage + 1,
        'feature': feature_name,
        'selected': item['mode'],
        'inactive_score': float(item['mode_scores']['inactive']),
        'eq_score': float(item['mode_scores']['target_value']),
    })

def print_table(records, columns, digits=3):
    widths = {}
    for column in columns:
        values = [column]
        for row in records:
            value = row[column]
            values.append(f'{value:.{digits}f}' if isinstance(value, float) else str(value))
        widths[column] = max(map(len, values))
    print('  '.join(column.ljust(widths[column]) for column in columns))
    for row in records:
        rendered = []
        for column in columns:
            value = row[column]
            text = f'{value:.{digits}f}' if isinstance(value, float) else str(value)
            rendered.append(text.ljust(widths[column]))
        print('  '.join(rendered))

print('archive demos:', len(bounds), 'source ids:', source_ids.tolist())
print()
print('MAP decisions')
print_table(mode_rows, ['user_stage', 'feature', 'selected', 'inactive_score', 'eq_score'], digits=6)
print()
print('Distribution summary in degrees')
print_table(summary_rows, ['user_stage', 'feature', 'demo_mean_mean_deg', 'between_demo_std_deg', 'demo_mean_range_deg', 'median_within_demo_std_deg', 'max_within_demo_std_deg', 'median_abs_drift_deg'])
print()
print('Per-demo stage-2 pitch')
print_table([r for r in rows if r['user_stage'] == 2 and r['feature'] == 'tool_pitch'], ['source_demo', 'n', 'mean_deg', 'std_deg', 'min_deg', 'max_deg', 'mean_error_deg', 'second_minus_first_deg'])
print()
print('Per-demo stage-4 roll')
print_table([r for r in rows if r['user_stage'] == 4 and r['feature'] == 'tool_roll'], ['source_demo', 'n', 'mean_deg', 'std_deg', 'min_deg', 'max_deg', 'mean_error_deg', 'second_minus_first_deg'])

archive demos: 12 source ids: [0, 1, 2, 3, 4, 5, 7, 8, 9, 10, 11, 12]

MAP decisions
user_stage  feature     selected      inactive_score  eq_score
2           tool_pitch  inactive      0.658935        0.337420
2           tool_roll   target_value  0.012369        0.960043
4           tool_pitch  target_value  0.069078        0.877579
4           tool_roll   inactive      0.999951        0.000000

Distribution summary in degrees
user_stage  feature     demo_mean_mean_deg  between_demo_std_deg  demo_mean_range_deg  median_within_demo_std_deg  max_within_demo_std_deg  median_abs_drift_deg
2           tool_pitch  90.007              2.892                 10.642               1.544                       4.587                    1.363               
2           tool_roll   -3.341              2.379                 7.009                0.995                       2.197                    0.947               
4           tool_pitch  93.400              3.050                 9.050             

In [4]:
import sys
sys.path.insert(0, str(root))
from envs.BarClean import BarCleanEnv, BarInspectScene

stage2_pitch_rows = [r for r in rows if r['user_stage'] == 2 and r['feature'] == 'tool_pitch']
stage4_roll_rows = [r for r in rows if r['user_stage'] == 4 and r['feature'] == 'tool_roll']

pitch_errors = np.asarray([r['mean_error_deg'] for r in stage2_pitch_rows])
pitch_ids = np.asarray([r['source_demo'] for r in stage2_pitch_rows])
worst_pitch = np.argsort(np.abs(pitch_errors))[::-1]
keep_without_two = np.ones(len(pitch_errors), dtype=bool)
keep_without_two[worst_pitch[:2]] = False

roll_means = np.asarray([r['mean_deg'] for r in stage4_roll_rows])
roll_ids = np.asarray([r['source_demo'] for r in stage4_roll_rows])

env = BarCleanEnv()
bar_poses = np.asarray(data['demo_bar_poses'], dtype=float)
bar_axis_yaws = []
for pose in bar_poses:
    scene = BarInspectScene(bar_pose_optitrack=pose, obstacle_pose_optitrack=np.array([0, 0, 0, 0, 0, 0, 1], dtype=float))
    geometry = env.get_top_view_scene_geometry()
    env.set_scene(scene)
    geometry = env.get_top_view_scene_geometry()
    axis = np.asarray(geometry['bar_axis_xy'], dtype=float)
    bar_axis_yaws.append(float(np.rad2deg(np.arctan2(axis[1], axis[0]))))
bar_axis_yaws = np.asarray(bar_axis_yaws)

print('Stage-2 pitch largest mean errors:', [(int(pitch_ids[i]), round(float(pitch_errors[i]), 3)) for i in worst_pitch[:4]])
print('Stage-2 pitch between-demo std, all:', round(float(np.std(pitch_errors)), 3), 'deg')
print('Stage-2 pitch between-demo std, excluding two worst:', round(float(np.std(pitch_errors[keep_without_two])), 3), 'deg')
print('Stage-2 pitch mean range, excluding two worst:', round(float(np.ptp(pitch_errors[keep_without_two])), 3), 'deg')
print()
print('Stage-4 roll correlation with source demo id:', round(float(np.corrcoef(roll_ids, roll_means)[0, 1]), 3))
print('Bar-axis yaw range across demos:', round(float(np.ptp(bar_axis_yaws)), 3), 'deg')
print('Stage-4 roll correlation with bar-axis yaw:', round(float(np.corrcoef(bar_axis_yaws, roll_means)[0, 1]), 3))
print()
print_table([
    {'source_demo': int(source_id), 'bar_axis_yaw_deg': float(axis_yaw), 'stage4_roll_mean_deg': float(roll)}
    for source_id, axis_yaw, roll in zip(source_ids, bar_axis_yaws, roll_means)
], ['source_demo', 'bar_axis_yaw_deg', 'stage4_roll_mean_deg'])

Stage-2 pitch largest mean errors: [(4, -6.141), (5, 4.501), (2, 4.087), (1, -3.971)]
Stage-2 pitch between-demo std, all: 2.892 deg
Stage-2 pitch between-demo std, excluding two worst: 2.052 deg
Stage-2 pitch mean range, excluding two worst: 8.059 deg

Stage-4 roll correlation with source demo id: -0.854
Bar-axis yaw range across demos: 1.091 deg
Stage-4 roll correlation with bar-axis yaw: -0.133

source_demo  bar_axis_yaw_deg  stage4_roll_mean_deg
0            -98.695           -0.070              
1            -98.625           6.321               
2            -98.594           -2.793              
3            -98.489           -2.741              
4            -98.756           -5.754              
5            -98.834           -5.980              
7            -98.861           -11.834             
8            -98.457           -9.860              
9            -97.770           -10.141             
10           -98.525           -10.833             
11           -98.653      